In [1]:

from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Any, Annotated, Literal
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
import operator
from langchain_anthropic import ChatAnthropic


load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("Anthropic_API_KEY", "")
 
model = ChatAnthropic(
    model="claude-sonnet-5-5",
    max_tokens=1000,
)



In [2]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = model.invoke(messages).text

    # response store state
    return {'messages': [response]}

In [4]:
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [5]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is **New Delhi**. It sits within the larger National Capital Territory of Delhi and serves as the seat of all three branches of the Indian government.'